In [12]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from IPython.display import display

load_dotenv()
DB_URL = os.getenv("DB_URL")

In [13]:
from utils.connect_db import connect_db

engine = connect_db(DB_URL)

2026-02-09 13:34:15,429 | INFO | utils.connect_db | Connected to PostgreSQL successfully


✅ Connected to PostgreSQL successfully!


In [14]:
from utils.ingestion import ingest_db

push_df = pd.read_csv("../dataset/Air-Clean.csv")
ingest_db(push_df, "flight_data", engine)

[INFO] Starting ingest into table 'flight_data' with 33734 rows


2026-02-09 13:34:17,213 | INFO | utils.ingestion | Appended 33734 rows into 'flight_data'


[SUCCESS] Appended 33734 rows into 'flight_data'


In [15]:
df_data_types = pd.read_sql("""SELECT
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_name = 'flight_data'
    ORDER BY ordinal_position;

""",engine)
display(df_data_types)

,column_name,data_type
0,airline,text
1,flightNumber,text
2,origin,text
3,destination,text
4,daysOfWeek,text
5,scheduledDepartureTime,time without time zone
6,scheduledArrivalTime,time without time zone
7,timezone,text
8,validFrom,date
9,validTo,date


In [16]:
df = pd.read_sql("""SELECT
       * FROM flight_data""",
    engine,
)
display(df.head())

,airline,flightNumber,origin,destination,daysOfWeek,scheduledDepartureTime,scheduledArrivalTime,timezone,validFrom,validTo,lastUpdated
0,GoAir,572,Goa,Hyderabad,"Thursday,Friday,Saturday",17:25:00,18:40:00,2021-03-27,2020-10-25,2021-03-27,2025-07-16
1,GoAir,301,Goa,Delhi,Saturday,12:00:00,14:45:00,2022-10-23,2022-03-27,2022-10-23,2025-07-16
2,GoAir,733,Goa,Nagpur,Saturday,12:10:00,13:40:00,2020-10-18,2020-03-29,2020-10-18,2025-07-16
3,GoAir,667,Goa,Lucknow,"Sunday,Monday,Tuesday,Wednesday,Thursday,Frida...",06:10:00,09:10:00,2019-10-26,2019-05-01,2019-10-26,2025-07-16
4,GoAir,247,Goa,Mumbai,Saturday,11:00:00,12:30:00,2022-03-20,2021-10-31,2022-03-20,2025-07-16


In [17]:
df.isna().sum()

airline                   0
flightNumber              0
origin                    0
destination               0
daysOfWeek                0
scheduledDepartureTime    0
scheduledArrivalTime      0
timezone                  0
validFrom                 0
validTo                   0
lastUpdated               0
dtype: int64

In [18]:
# Same origin & destination
invalid_routes = df[df['origin'] == df['destination']]
invalid_routes.shape

(0, 11)

In [19]:
df = df[df['origin'] != df['destination']]

In [20]:
(df['validTo'] < df['validFrom']).sum()


np.int64(0)

In [21]:
df['airline'].value_counts()


airline
IndiGo                  305056
SpiceJet                 53968
Air India                49872
GoAir                    29664
Air India Express        28848
Vistara                  24240
Alliance Air (India)     23888
Jet Airways               7376
Akasa Air                 6608
TruJet                    4816
Star Air                  2992
FlyBig                    1664
Jetlite                    656
IndiaOne Air                96
Name: count, dtype: int64

In [22]:
df['days_list'] = df['daysOfWeek'].str.split(',')
df['weekly_frequency'] = df['days_list'].apply(len)


In [23]:
df['weekly_frequency'] = df['days_list'].apply(len)


In [24]:
# Weekday vs Weekend flag
weekend_days = {'Saturday', 'Sunday'}

df['has_weekend_ops'] = df['days_list'].apply(
    lambda days: any(d in weekend_days for d in days)
)

df['dep_time'] = pd.to_datetime(df['scheduledDepartureTime'], format='%H:%M:%S')
df['arr_time'] = pd.to_datetime(df['scheduledArrivalTime'], format='%H:%M:%S')
df['valid_from'] = pd.to_datetime(df['validFrom'], errors='coerce')


In [25]:
# Convert time → minutes since midnight
dep_min = (
    df["scheduledDepartureTime"].apply(lambda t: t.hour * 60 + t.minute)
)

arr_min = (
    df["scheduledArrivalTime"].apply(lambda t: t.hour * 60 + t.minute)
)

# Duration
df["duration_min"] = arr_min - dep_min

# Handle overnight flights
df.loc[df["duration_min"] < 0, "duration_min"] += 1440


In [26]:
df['route'] = df['origin'] + ' → ' + df['destination']

In [27]:
def dep_bucket(hour):
    if hour < 6:
        return 'Early Morning'
    elif hour < 12:
        return 'Morning'
    elif hour < 18:
        return 'Afternoon'
    else:
        return 'Night'

df['dep_bucket'] = df['dep_time'].dt.hour.apply(dep_bucket)

In [28]:
def indian_season(month):
    if month in [1, 2]:
        return 'Winter'
    elif month == 3:
        return 'Pre-Summer'
    elif month in [4, 5, 6]:
        return 'Summer'
    elif month in [7, 8, 9]:
        return 'Monsoon'
    elif month == 10:
        return 'Post-Monsoon'
    else:  # 11, 12
        return 'Winter'

df['season'] = df['valid_from'].dt.month.apply(indian_season)
df

,airline,flightNumber,origin,destination,daysOfWeek,scheduledDepartureTime,scheduledArrivalTime,timezone,validFrom,validTo,...,days_list,weekly_frequency,has_weekend_ops,dep_time,arr_time,valid_from,duration_min,route,dep_bucket,season
0,GoAir,572,Goa,Hyderabad,"Thursday,Friday,Saturday",17:25:00,18:40:00,2021-03-27,2020-10-25,2021-03-27,...,"[Thursday, Friday, Saturday]",3,True,1900-01-01 17:25:00,1900-01-01 18:40:00,2020-10-25,75,Goa → Hyderabad,Afternoon,Post-Monsoon
1,GoAir,301,Goa,Delhi,Saturday,12:00:00,14:45:00,2022-10-23,2022-03-27,2022-10-23,...,[Saturday],1,True,1900-01-01 12:00:00,1900-01-01 14:45:00,2022-03-27,165,Goa → Delhi,Afternoon,Pre-Summer
2,GoAir,733,Goa,Nagpur,Saturday,12:10:00,13:40:00,2020-10-18,2020-03-29,2020-10-18,...,[Saturday],1,True,1900-01-01 12:10:00,1900-01-01 13:40:00,2020-03-29,90,Goa → Nagpur,Afternoon,Pre-Summer
3,GoAir,667,Goa,Lucknow,"Sunday,Monday,Tuesday,Wednesday,Thursday,Frida...",06:10:00,09:10:00,2019-10-26,2019-05-01,2019-10-26,...,"[Sunday, Monday, Tuesday, Wednesday, Thursday,...",7,True,1900-01-01 06:10:00,1900-01-01 09:10:00,2019-05-01,180,Goa → Lucknow,Morning,Summer
4,GoAir,247,Goa,Mumbai,Saturday,11:00:00,12:30:00,2022-03-20,2021-10-31,2022-03-20,...,[Saturday],1,True,1900-01-01 11:00:00,1900-01-01 12:30:00,2021-10-31,90,Goa → Mumbai,Morning,Post-Monsoon
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539739,IndiGo,959,Bengaluru,Indore,"Monday,Wednesday,Friday",06:20:00,08:15:00,2024-03-30,2023-10-31,2024-03-30,...,"[Monday, Wednesday, Friday]",3,False,1900-01-01 06:20:00,1900-01-01 08:15:00,2023-10-31,115,Bengaluru → Indore,Morning,Post-Monsoon
539740,IndiGo,5998,Bengaluru,Allahabad,"Tuesday,Thursday,Saturday",10:25:00,13:00:00,2021-03-27,2020-10-25,2021-03-27,...,"[Tuesday, Thursday, Saturday]",3,True,1900-01-01 10:25:00,1900-01-01 13:00:00,2020-10-25,155,Bengaluru → Allahabad,Morning,Post-Monsoon
539741,IndiGo,6035,Bengaluru,Allahabad,"Sunday,Tuesday,Thursday,Saturday",08:40:00,10:55:00,2024-10-25,2024-03-31,2024-10-25,...,"[Sunday, Tuesday, Thursday, Saturday]",4,True,1900-01-01 08:40:00,1900-01-01 10:55:00,2024-03-31,135,Bengaluru → Allahabad,Morning,Pre-Summer
539742,IndiGo,6514,Bengaluru,Chandigarh,Monday,09:40:00,13:00:00,2019-10-22,2019-04-02,2019-10-22,...,[Monday],1,False,1900-01-01 09:40:00,1900-01-01 13:00:00,2019-04-02,200,Bengaluru → Chandigarh,Morning,Summer


In [29]:
analysis_df = df[[
    'airline',
    'flightNumber',
    'origin',
    'destination',
    'scheduledDepartureTime',
    'scheduledArrivalTime',
    'route',
    'weekly_frequency',
    'has_weekend_ops',
    'dep_bucket',
    'duration_min',
    'season',
    'valid_from',
    'validTo'
]].copy()

analysis_df.rename(columns={
    'flightNumber': 'flight_number',
    'scheduledDepartureTime': 'scheduled_departure_time',
    'scheduledArrivalTime': 'scheduled_arrival_time',
    'validTo': 'valid_to'
}, inplace=True)

In [30]:
ingest_db(analysis_df, "analyze_data", engine)


[INFO] Starting ingest into table 'analyze_data' with 539744 rows


2026-02-09 13:35:02,993 | INFO | utils.ingestion | Appended 539744 rows into 'analyze_data'


[SUCCESS] Appended 539744 rows into 'analyze_data'
